# 03 — Job Recommender (Core Engine, Unsupervised)

Content-based recommendation: vectorize every job posting + the resume with TF-IDF, rank by
cosine similarity, return the top-N. This is the heart of the SmartHire portal.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd

from src.data import load_data, preprocess
from src.models import recommender
from src import config

## 1. Load (or build) the job corpus

In [2]:
naukri = load_data.load_naukri()
linkedin = load_data.load_linkedin()
job_corpus = preprocess.build_job_corpus(naukri, linkedin)
job_corpus.shape

(58089, 16)

## 2. Build the TF-IDF job index

In [3]:
job_vectorizer, job_matrix = recommender.build_job_index(job_corpus)
print(job_matrix.shape)

(58089, 25000)


In [4]:
recommender.save_job_index(job_vectorizer, job_matrix)

## 3. Try a recommendation for a sample resume

In [5]:
resumes_raw = load_data.load_resumes()
resumes_clean = preprocess.clean_resumes(resumes_raw)

sample = resumes_clean[resumes_clean['category'] == resumes_clean['category'].iloc[0]].iloc[0]
print('Category:', sample['category'])
print(sample['resume_text'][:400])

Category: HR
TECHNICAL SKILLS â¢ Typewriting â¢ TORA â¢ SPSSEducation Details 
January 2017 MBA  Chidambaram, Tamil Nadu SNS College of Engineering
January 2014 HSC   at SAV Higher Secondary School
 MBA   SNS College of Engineering
 SSLC Finance  at Kamaraj Matriculation School
HR 


Skill Details 
Human resource, Finance- Exprience - Less than 1 year monthsCompany Details 
company - 
description


In [6]:
top_matches = recommender.recommend_jobs(sample['clean_text'], job_corpus, job_vectorizer, job_matrix, top_n=10)
top_matches[['title', 'company', 'source', 'match_score']]

,title,company,source,match_score
0,"RF Engineer (Karnataka, Telangana,Tamil Nadu)",3D Structure Multimedia,naukri,25.0
1,Amazon is Hiring For International Voice - Tam...,Amazon,naukri,24.5
2,Urgently Required Service Engineer @ Tamil Nadu !,Tectronics Engineers,naukri,24.2
3,Area Sales Manager - MNC FMCG CO - Open Tamil ...,,linkedin,23.9
4,Amazon is Hiring For International Voice - Tam...,Amazon,naukri,22.2
5,Senior NABH Consultant for Chennai & Tamil Nadu,,linkedin,20.2
6,G.M - Sales - Tamil Nadu,,linkedin,20.1
7,Service Engineer,Building Control Solutions,naukri,16.9
8,Human Resource (HR),,linkedin,16.7
9,Senior Role - Human Resource - Food,,linkedin,15.8


## 4. Qualitative check across categories
For each category, do the top-10 recommended job titles plausibly relate to that category's keyword?

In [6]:
rows = []
for cat in resumes_clean['category'].unique():
    text = resumes_clean[resumes_clean['category'] == cat]['clean_text'].iloc[0]
    p10 = recommender.precision_at_k(job_corpus, job_vectorizer, job_matrix, 'category', cat, text, k=10)
    rows.append({'category': cat, 'precision_at_10': p10})

pd.DataFrame(rows).sort_values('precision_at_10', ascending=False)

,category,precision_at_10
1,Java Developer,1.0
4,DESIGNER,0.8
6,TEACHER,0.8
19,ACCOUNTANT,0.7
2,DevOps Engineer,0.4
14,DIGITAL-MEDIA,0.4
12,SALES,0.3
17,APPAREL,0.2
23,ARTS,0.1
22,BANKING,0.1


## Notes on Precision@K here

There's no ground-truth "this job is relevant to this resume" label in the raw data, so
`precision_at_k` uses a simple keyword heuristic (does the recommended job title contain a
word from the resume's category?) as a proxy for relevance. It's a sanity check, not a rigorous
metric — the qualitative spot-checks above matter just as much.